# Дообучение YOLOv8 для видеонаблюдения (Google Colab)

Ноутбук обучает YOLOv8n на 5 классах Pascal VOC, важных для систем безопасности
(person, car, bus, motorbike, bicycle), считает метрики качества (Precision, Recall, mAP@0.5)
и запускает трекинг объектов на видео.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**. Полный прогон занимает около часа.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Код проекта и зависимости

In [ ]:
!git clone -q https://github.com/NurimanB/intelligent-security-cv.git
%cd intelligent-security-cv
!pip install -q -r requirements.txt

## 2. Обучение

Датасет VOC (~3 ГБ) скачивается автоматически при первом запуске.
Для быстрой проверки, что всё работает, можно сначала запустить:
`!python scripts/train.py --data coco8.yaml --epochs 1 --classes`

In [ ]:
!python scripts/train.py --data VOC.yaml --model yolov8n.pt --epochs 30 --fraction 0.5 --device 0

## 3. Метрики качества

In [ ]:
import json
from IPython.display import Image, display

metrics = json.load(open("reports/metrics.json"))
weights = metrics["weights"]
print(f"mAP@0.5      = {metrics['mAP50']:.3f}")
print(f"mAP@0.5:0.95 = {metrics['mAP50_95']:.3f}")
print(f"Precision    = {metrics['precision']:.3f}")
print(f"Recall       = {metrics['recall']:.3f}")
print("AP@0.5 по классам:", metrics["per_class_AP50"])

display(Image("reports/figures/train_results.png", width=900))
display(Image("reports/figures/train_confusion_matrix.png", width=600))

## 4. Трекинг объектов на видео

Загрузите любое короткое видео (улица, парковка, вход в здание) — каждый объект получит уникальный `track_id`.

In [ ]:
from google.colab import files

uploaded = files.upload()
video_path = next(iter(uploaded))

!python scripts/track_video.py --source "{video_path}" --weights "{weights}"

In [ ]:
import pandas as pd

tracks = pd.read_csv("data/processed/tracks.csv")
tracks.groupby("object_class")["track_id"].nunique().sort_values(ascending=False)

## 5. Сохранить результаты

Скачайте веса и метрики. Добавьте `reports/metrics.json` и графики из `reports/figures/` в репозиторий —
по ним заполняется таблица результатов в README.

In [ ]:
from google.colab import files

!zip -q -r results.zip reports "{weights}"
files.download("results.zip")